This notebook is used to get the Experiment results for the dropclass-publication

# Prerequisites

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

## Main df

In [2]:
# Import data
df_main = pd.read_excel('../data/df_main.xlsx', index_col=0).reset_index(drop=True)
## Drop unused columns
# drop_columns = ['voltage', 'long_impulse_duration', 'long_impulse_dur_binary']
# df_main = df_main.drop(drop_columns, axis=1)
display(df_main.head())
df_main.info()

,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,roughness,long_impulse_dur_binary,roughness_binary,volume_fraction_binary,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat
0,3,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
1,4,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
2,5,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
3,7,0,1,0,0,0,105.0,10,0.8,0,...,0.04,low,0,1,3.961141,1435.111557,1434.906112,233.148786,0.013833,small
4,8,0,1,0,0,0,105.0,10,0.8,0,...,0.04,low,0,1,3.961141,1435.111557,1434.906112,233.148786,0.013833,small


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   test                             372 non-null    int64  
 1   one_drop                         372 non-null    int64  
 2   splashing                        372 non-null    int64  
 3   breaking_up                      372 non-null    int64  
 4   net_impact                       372 non-null    int64  
 5   rebound                          372 non-null    int64  
 6   voltage                          372 non-null    float64
 7   long_impulse_duration            372 non-null    int64  
 8   height                           372 non-null    float64
 9   inclination                      372 non-null    int64  
 10  droplet_diameter                 372 non-null    float64
 11  liquid_density                   372 non-null    int64  
 12  surface_tension       

In [3]:
label_name = ['net_impact', 'splashing', 'breaking_up', 'rebound']

class_count_dict = {}

for label in label_name:
    class_count_dict[label] = df_main[df_main[label]==1].shape[0]

class_count_s = pd.Series(class_count_dict)
class_count_s

net_impact     121
splashing      174
breaking_up    101
rebound         99
dtype: int64

**Labels description:**
- **splashing**: when *'Number of detached small droplets during Spreading'=='many' (more than 5)*;
- **breaking_up**: combines clear "breaking_up" during receding *'Droplet Receding'==2*, as well as the Spreading detaching *0<'Number of detached small droplets during spreading'<=5*;
- **rebound**: combines all rebound types (partial and total, *'Rebound'>0*), as well as the central jet ejection *'Rim merging or Central jet ejecting'==2*
- **net_impact**: when there is 
    - **no Splashing** (no small droplets detached during spreading, *'Number of detached small droplets during Spreading'==0*), 
    - **no Breaking up** (*'Droplet Receding'<2* [equivalent is *'Number of detached small droplets during Receding or Rim merging' == 0*])
    - **no Rebound** (*'Rebound' == 0*)

**Total count of clear experiments is *372***

## Suspension Data Labeling Edited. For the classes reconfiguration

In [4]:
df_labels = pd.read_excel('../data/Suspension Data Labeling Edited.xlsx', index_col=0)
df_labels = df_labels.drop(index=0)
# Keep only used 372 tests (without draft substrated and liquids)
df_labels = df_labels[df_labels['Test #'].isin(df_main.test.values)]
df_labels = df_labels.drop('Comments', axis=1)
df_labels = df_labels.rename({'Test #': 'test'}, axis=1)
df_labels['test'] = df_labels['test'].astype(int)
df_labels = df_labels.reset_index(drop=True)
display(df_labels)
display(df_labels.info())
df_labels.columns

,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
0,3,1,many,0,0,0,0,0,1
1,4,1,many,0,0,0,0,0,1
2,5,1,many,0,0,0,0,0,1
3,7,0,many,1,0,0,0,0,2
4,8,0,many,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...
367,391,0,0,1,2,0,2,0,0
368,392,1,3,1,1,0,0,0,1
369,393,1,2,1,1,0,0,0,3
370,394,1,0,1,1,1,0,0,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 9 columns):
 #   Column                                                            Non-Null Count  Dtype 
---  ------                                                            --------------  ----- 
 0   test                                                              372 non-null    int64 
 1   Total number of dropped drops                                     372 non-null    object
 2   Number of detached small droplets during Spreading                372 non-null    object
 3   Droplet Receding                                                  372 non-null    object
 4   Rim merging or Central jet ejecting                               372 non-null    object
 5   Number of detached small droplets during Receding or Rim merging  372 non-null    object
 6   Rebound                                                           372 non-null    object
 7   Number of detached droplets during Rebound   

None

Index(['test', 'Total number of dropped drops',
       'Number of detached small droplets during Spreading',
       'Droplet Receding', 'Rim merging or Central jet ejecting',
       'Number of detached small droplets during Receding or Rim merging',
       'Rebound', 'Number of detached droplets during Rebound',
       'Final droplets count'],
      dtype='object')


| Parameter name | Description | Possible values |
| --- | --- | --- |
| Total number of dropped drops | The total number of drops that fell from above | 1, 2 |
| Number of detached small droplets during *spreading* | The number of small droplets that flew out of the lamella drops in the process of spreading | 0, 1, .., 5, many |
| Droplet Receding | Presence (1,2), absence — 0 of a visible drop shrinkage, and an increase in the drop height after reaching the maximum spot diameter. 1 — the usual reduction of the main drop, 2 — the destruction of the main drop in the process of reduction | 0, 1, 2 |
| Rim merging or Central jet ejecting | The presence (1,2), absence — 0 of an explicit rim merging at the end of the shrinkage of the drop (1) and/or the reverse vertical jet (2) (against gravity) | 0, 1, 2 |
| Number of detached small droplets during *Receding or Rim merging* | The number of small droplets separating from the rim of the main drop (rim) and remaining on the substrate during the shrinkage of the drop (receding). Drops do not separate — 0, several drops, many drops — many | 0, 1, .., 5, many |
| Rebound | Presence (1,2), absence — 0 of a drop rebound after the formation of a reverse vertical jet (central jet ejecting). Partial rebound — 1, total rebound - 2 | 0, 1, 2 |
| Number of detached droplets during *Rebound* | The number of droplets separating from the reverse vertical jet (central jet) | 0, 1, .., 5, many |
| Final droplets count | The total number of droplets on the substrate, clearly separated from each other | 1, .., 5, many |

# Classes Research

## Break up dividing. Create 'breaking_up_receding' and 'splashing_semi'

How many receding break up tests, when *'Number of detached small droplets during Receding or Rim merging' > 0*

In [5]:
index_receding_break_up = df_labels[df_labels['Number of detached small droplets during Receding or Rim merging'] != 0].index
receding_break_up_count = index_receding_break_up.shape[0]
print(f'Receding break_up tests count: {receding_break_up_count}')

Receding break_up tests count: 60


Let us check, that not_receding_break_up cases are happends only if number of detached small droplets during spreading == 'few'

In [6]:
# Not receding break up, when label is 'breaking_up', but no detached small droplets during Receding or Rim merging
index_not_receding_break_up = df_main[(~df_main.index.isin(index_receding_break_up))&(df_main['breaking_up']==1)].index
not_receding_break_up_count = index_not_receding_break_up.shape[0]
print(f'Not receding break_up tests count: {not_receding_break_up_count}')

Not receding break_up tests count: 41


In [7]:
print('Not receding break up labels')
not_receding_break_up_labels = df_labels.loc[index_not_receding_break_up]
display(not_receding_break_up_labels['Number of detached small droplets during Spreading'].value_counts())
display(not_receding_break_up_labels['Number of detached small droplets during Receding or Rim merging'].value_counts())
display(not_receding_break_up_labels)

Not receding break up labels


Number of detached small droplets during Spreading
2    15
1     9
3     8
4     5
5     4
Name: count, dtype: int64

Number of detached small droplets during Receding or Rim merging
0    41
Name: count, dtype: int64

,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
17,26,1,2,0,0,0,0,0,1
18,27,0,5,0,0,0,0,0,1
38,51,0,2,0,0,0,0,0,1
46,59,1,1,1,1,0,0,0,1
47,60,1,1,1,0,0,0,0,1
58,74,1,1,0,0,0,0,0,1
60,78,1,2,1,0,0,0,0,2
63,81,0,5,0,0,0,0,0,2
78,100,1,3,1,1,0,0,0,1
86,109,1,1,1,1,0,0,0,1


Thus, only 'few small droplets during Spreading' cases are not in receding break_up

**Thus, let us create a receding break up** instead of *break up*

In [8]:
df_main['breaking_up_receding'] = 0
df_main.loc[index_receding_break_up, 'breaking_up_receding'] = 1
df_main['breaking_up_receding'].value_counts()

breaking_up_receding
0    312
1     60
Name: count, dtype: int64

Now, let us check classes of tests with a few count of 'Number of detached small droplets during Spreading'

In [9]:
index_spreading_few_detached = df_labels[df_labels['Number of detached small droplets during Spreading'].isin(list(range(1,6)))].index
# Show tests with few drops detached during Spreading
df_main.iloc[index_spreading_few_detached]

,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,long_impulse_dur_binary,roughness_binary,volume_fraction_binary,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat,breaking_up_receding
9,17,0,0,1,0,0,103.0,21,0.8,0,...,high,0,1,3.961141,11559.676628,650.561372,264.472523,0.045305,medium,1
10,18,0,0,1,0,0,103.0,18,0.8,0,...,high,0,1,3.961141,11559.676628,650.561372,264.472523,0.045305,medium,1
17,26,1,0,1,0,0,108.0,10,0.8,0,...,low,0,1,3.961141,11426.368989,643.059016,262.181762,0.045833,medium,0
18,27,0,0,1,0,0,108.0,10,0.8,0,...,low,0,1,3.961141,11883.423749,668.781377,270.008528,0.044071,medium,0
38,51,0,0,1,0,0,106.5,12,0.8,0,...,low,0,1,3.961141,677.852440,913.477171,154.217138,0.012388,small,0
42,55,0,0,1,0,1,111.0,12,0.8,0,...,low,0,1,3.961141,631.313317,850.760828,146.205928,0.013301,small,1
46,59,1,0,1,0,0,107.0,12,0.8,0,...,low,0,1,3.961141,693.028241,933.928153,156.799425,0.012117,small,0
47,60,1,0,1,0,0,107.0,12,0.8,0,...,low,0,1,3.961141,693.028241,933.928153,156.799425,0.012117,small,0
58,74,1,0,1,0,0,101.0,21,0.8,0,...,high,1,1,3.961141,560.088225,754.777555,133.651427,0.014993,small,0
60,78,1,0,1,0,0,108.0,15,0.8,0,...,high,0,1,3.961141,687.969640,927.111159,155.940249,0.040441,medium,0


Let us introduce 'semi-splashing' as class for the few detached droplets

In [10]:
df_main['splashing_semi'] = 0
df_main.loc[index_spreading_few_detached, 'splashing_semi'] = 1
df_main['splashing_semi'].value_counts()

splashing_semi
0    327
1     45
Name: count, dtype: int64

## Rebound dividing. Create 'rebound_total', 'rebound_true', 'jet_ejection'

In [11]:
df_main['rebound'].value_counts()

rebound
0    273
1     99
Name: count, dtype: int64

In [12]:
index_rebound_total = df_labels[df_labels['Rebound'] == 2].index
print(f'Real total rebound cases count: {index_rebound_total.shape[0]}')
display(df_labels.loc[index_rebound_total])

Real total rebound cases count: 38


,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
69,90,0,many,2,2,many,2,0,many
71,92,0,many,2,2,many,2,0,many
75,96,1,many,2,2,many,2,0,many
99,122,1,many,2,2,many,2,0,many
100,123,0,many,2,2,many,2,0,many
101,124,1,many,2,2,many,2,0,many
110,133,1,many,1,2,many,2,0,many
174,197,1,many,2,2,many,2,0,many
176,199,0,many,2,2,many,2,0,many
177,200,0,many,2,2,many,2,0,many


According to videos of tests, when Rebound==2, but Number of detached droplets during Rebound == 0 - there is no mistake, so let us include 'rebound_total' for the analysis

In [13]:
df_main['rebound_total'] = 0
df_main.loc[index_rebound_total, 'rebound_total'] = 1
df_main['rebound_total'].value_counts()

rebound_total
0    334
1     38
Name: count, dtype: int64

Let us consider partial rebound cases

In [14]:
index_rebound_partial = df_labels[(df_labels['Rebound'] == 1)&(df_labels['Number of detached droplets during Rebound']>0)].index
print(f'Number of partial rebound cases {index_rebound_partial.shape[0]}')
df_labels.loc[index_rebound_partial]

Number of partial rebound cases 6


,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
14,23,0,many,2,2,many,1,1,many
33,45,1,many,1,2,many,1,1,many
274,298,1,1,1,2,0,1,1,1
282,306,1,0,1,2,0,1,1,1
291,315,1,many,1,2,0,1,3,1
294,318,0,many,1,2,0,1,2,1


Only 6 cases has real partial rebound, let us combine total and partial rebound -> 'rebound_true'

In [15]:
index_rebound_true = list(set(index_rebound_total).union(set(index_rebound_partial)))

df_main['rebound_true'] = 0
df_main.loc[index_rebound_true, 'rebound_true'] = 1
df_main['rebound_true'].value_counts()

rebound_true
0    328
1     44
Name: count, dtype: int64

Now, let us check not true rebound

In [16]:
index_not_rebound = df_main[(df_main['rebound_true']==0)&(df_main['rebound']==1)].index
print(f'Number of not rebound cases: {index_not_rebound.shape[0]}')
display(df_main.loc[index_not_rebound])
display(df_labels.loc[index_not_rebound])
display(df_labels.loc[index_not_rebound]['Rim merging or Central jet ejecting'].value_counts())

Number of not rebound cases: 55


,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat,breaking_up_receding,splashing_semi,rebound_total,rebound_true
12,20,0,1,1,0,1,103.0,15,0.8,0,...,3.961141,11864.379801,667.709612,269.683934,0.044141,medium,1,0,0,0
13,22,0,1,1,0,1,110.5,12,0.8,0,...,3.961141,12340.478509,694.503738,277.760377,0.042438,medium,1,0,0,0
23,34,1,1,1,0,1,105.0,12,0.8,0,...,3.961141,6615.685566,753.458811,247.556009,0.012848,small,1,0,0,0
24,35,0,1,1,0,1,105.0,12,0.8,0,...,3.961141,6390.383581,727.799223,241.205677,0.013301,small,1,0,0,0
32,44,1,1,0,0,1,120.0,12,0.8,0,...,3.961141,2172.648606,858.125870,199.996719,0.012519,small,0,0,0,0
42,55,0,0,1,0,1,111.0,12,0.8,0,...,3.961141,631.313317,850.760828,146.205928,0.013301,small,1,1,0,0
43,56,0,0,1,0,1,106.0,12,0.8,0,...,3.961141,677.852440,913.477171,154.217138,0.012388,small,1,0,0,0
53,69,1,0,0,0,1,108.0,15,0.8,0,...,3.961141,682.304008,919.476126,154.976093,0.012307,small,0,0,0,0
54,70,1,0,1,0,1,103.5,21,0.8,0,...,3.961141,682.304008,919.476126,154.976093,0.012307,small,1,0,0,0
62,80,1,1,1,0,1,108.0,15,0.8,0,...,3.961141,661.664919,891.662791,151.446712,0.042049,medium,1,0,0,0


,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
12,20,0,many,2,2,many,1,0,many
13,22,0,many,2,2,many,1,0,many
23,34,1,many,2,2,many,1,0,many
24,35,0,many,2,2,many,1,0,many
32,44,1,many,1,2,0,1,0,many
42,55,0,1,1,2,3,1,0,3
43,56,0,0,1,2,2,1,0,2
53,69,1,0,1,2,0,1,0,1
54,70,1,0,1,2,1,1,0,1
62,80,1,many,1,2,many,1,0,many


Rim merging or Central jet ejecting
2    55
Name: count, dtype: int64

We see, that all not rebound cases has the jet ejection

Now, let us create label - jet_ejection

In [17]:
df_main['jet_ejection'] = 0
df_main.loc[index_not_rebound, 'jet_ejection'] = 1
df_main['jet_ejection'].value_counts()

jet_ejection
0    317
1     55
Name: count, dtype: int64

## Splashing

This class is the same, since it has simple condition: 'Number of detached droplets during spreading' > 5. Total count = 174

In [18]:
index_splashing = df_main[df_main['splashing'] == 1].index

df_labels.loc[index_splashing, 'Number of detached small droplets during Spreading'].value_counts()

Number of detached small droplets during Spreading
many    173
7         1
Name: count, dtype: int64

## Net impact

Let us check, that net impact occurs only when other new labels are 0

In [19]:
df_main[(df_main['splashing'] == 0) & (df_main['splashing_semi'] == 0) & (df_main['rebound_true'] == 0) & (df_main['breaking_up_receding'] == 0) & (df_main['jet_ejection'] == 0)]['net_impact'].value_counts()

net_impact
1    121
Name: count, dtype: int64

The number is the same as in the original classification

## Save result

Now we have edited main dataset with new labels and old classes. Let us save it

In [20]:
# df_main.to_excel('../data/df_main_edited.xlsx', index=False)

# Experiments Research

In [ ]:
## Drop unused columns
drop_columns = ['voltage', 'long_impulse_duration', 'long_impulse_dur_binary']
df_main = df_main.drop(drop_columns, axis=1)